# Load minIO bronze to MinIO silver (Incremental)

### Install python dotenv for get the environment variables

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Imports libs, files and configure the absolute path

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession, functions
from pyspark.sql.functions import date_format
import logging
from datetime import datetime

# Import for get the environment variables 
from dotenv import load_dotenv
import os
import sys
sys.path.append(os.path.abspath("../../")) # Used to reconfigure the absolut path. In this case, setting the absolut path to 2 folders back (notebooks/...) 
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

## Load environment variables

In [2]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")

## Spark configurations

In [3]:
conf = SparkConf()

conf.setAppName("Incremental transform from MinIO bronze to MinIO silver") # Spark application name, Usefull for logs
# Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
# add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
conf.set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,"
         "com.amazonaws:aws-java-sdk-bundle:1.12.767,"
         "org.postgresql:postgresql:42.7.2,"
         "io.delta:delta-spark_2.12:3.2.0")
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).getOrCreate()

## Configuration the Logging and log the startup

In [4]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Logging the Start process from ingestion
logging.info("Starting incrmental transform from MinIO bronze to MinIO silver...")

2026-05-08 19:56:37,423 - INFO - Starting incrmental transform from MinIO bronze to MinIO silver...


In [6]:
for table_name in config_file.queries_silver.keys():
    
    try:
        queries_tables = config_file.queries_silver
        
        # bronze path
        bronze_path = config_file.data_lakehouse_path["bronze"]
        # silver path
        silver_path = config_file.data_lakehouse_path["silver"]
        
        # bronze table path
        bronze_table_path = f"{bronze_path}bronze_{table_name}"
        # silver table path
        silver_table_path = f"{silver_path}silver_{table_name}"
        
        # Logging the processing
        logging.info(f"processing table {table_name}")
        
        # Getting max date value from minIO silver in the modifieddate column. limit at 1 result and get this result on 1º row at max_modifieddate column
        logging.info("Reading table from silver layer")
        df_max_modifieddate_silver = spark.read.format("delta").load(silver_table_path) \
            .select(functions.max("modifieddate").alias("max_modifieddate")).limit(1).collect()[0]["max_modifieddate"]

        query = func_file.get_query(table_name, queries_tables, bronze_path)
        
         #Transforming data from the bronze layer where the “modifieddate” column is more recent than the “modifieddate” column in the silver layer
        query_update_data_from_silver = spark.sql(f"""
            select * from ({query}) as subquery
            where modifieddate > '{df_max_modifieddate_silver}'
            """)

        # Number of rows returns from query to update, if exists
        rows_to_update = query_update_data_from_silver.count()
        
        if rows_to_update == 0:
            # Logging if get no rows to update in minio bronze
            logging.info(f"No new data to process for table {table_name}")

        else:
            # Logging number of rows to update
            logging.info(f"Number of new rows to update for table {table_name}: {rows_to_update}")
            
            # Adding a new column date related the load data
            df_with_update_date = func_file.add_data_last_update(query_update_data_from_silver)
    
            # modifing dataframe to add a new column "month_key" to create a partition on the minIO silver based on modifieddate column
            df_with_month_partition = df_with_update_date.withColumn("month_key", date_format(df_with_update_date["modifieddate"], "yyyy-MM"))
            
            # Updating the dataframe on minIO silver
            logging.info(f"Updating table {table_name}...")
            df_with_month_partition.write.format("delta").mode("append").partitionBy("month_key").save(silver_table_path)
            
            # Logging the sucessfully process
            logging.info(f"Table {table_name} Sucessfully updated and saved in MinIO silver on: {silver_table_path}")

    except Exception as e:
        # Logging the Error
         logging.error(f"Error processing table {table_name}: {str(e)}")

# Logging the Incremental ingestion
logging.info(f"Incremental ingestion to silver layer completed!")
    

2026-05-08 19:38:29,050 - INFO - processing table sales_countryregioncurrency
2026-05-08 19:39:03,106 - INFO - No new data to process for table sales_countryregioncurrency
2026-05-08 19:39:03,107 - INFO - processing table sales_creditcard
2026-05-08 19:39:10,378 - INFO - No new data to process for table sales_creditcard
2026-05-08 19:39:10,380 - INFO - processing table sales_currency
2026-05-08 19:39:15,206 - INFO - No new data to process for table sales_currency
2026-05-08 19:39:15,208 - INFO - processing table sales_currencyrate
2026-05-08 19:39:20,891 - INFO - No new data to process for table sales_currencyrate
2026-05-08 19:39:20,894 - INFO - processing table sales_customer
2026-05-08 19:39:23,669 - INFO - No new data to process for table sales_customer
2026-05-08 19:39:23,670 - INFO - processing table sales_personcreditcard
2026-05-08 19:39:26,919 - INFO - No new data to process for table sales_personcreditcard
2026-05-08 19:39:26,920 - INFO - processing table sales_salesorderdeta